***
# **Parte B: Clasificación**

El regresor lineal permite predecir un valor continuo. En el caso que quisieramos utilizar el regresor en una tarea de clasificación, será necesario acoplar a la salida del regresor una *función de activación de tipo sigmoide*.



### Regresor Logístico:

El nuevo modelo se denomina **regresor logísitico**, y su salida no es la predicción de una clase sino que puede interpretarse como una `probabilidad`.

La *probabilidad posterior* de que los datos ($x$) pertenezcan a la clase ($C1$) se define como:

$$P(C1|x)= sigmoid(xw + b)$$

Para el caso binario (2 clases solamente), ocurre que:

$$P(C2|x)= 1 - P(C1|x)$$

Ahora nuestro objetivo será ajustar o aprender los parámetros del modelo ($w$ y $b$) que clasifiquen mejor los datos de entrenamiento. 

La función de activación `sigmoide` se define como:

$$sigmoid(z)=\frac{1}{1+e^{-z}}$$



### Binary Cross-Entropy

La función de pérdida que utilizaremos es la **binary cross-entropy** (BCE) o  entropía cruzada binaria entre el valor esperado (target, $y$) y el valor predicho por el modelo ($\hat{y}$):

$$BCE(y,\hat{y})=\frac{\sum(-y*log(\hat{y})-(1-y)*log(1-\hat{y}))}{N}$$

****
#### **Ejercicio B-1**:

* Utilizaremos el algoritmo de descenso de gradiente para encontrar el mínimo de dicha función.

* Emplearemos  un dataset del área médica, con datos sobre `cancer de pecho`, generado por investigadores de la Universidad de Wisconsin y provisto por la [UCI](https://archive.ics.uci.edu/ml/datasets/Breast+Cancer+Wisconsin+(Diagnostic)). 

* Para acceder al dataset no es necesario descargarlo ya que se encuentra provisto por el módulo [_sklearn.datasets_](http://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html#sklearn.datasets.load_breast_cancer) de la librería **scikit-learn** que se utilizará durante el curso.

* El dataset tiene 569 instancias con 30 atributos (features) cada una. Las instancias o muestras (samples) pueden ser clasificadas entre **Malignos** y **Benignos** (`clasificación binaria`). 

* El dataset está ligeramente desbalanceado, lo que significa que existen más instancias de una clase que de la otra. En partícular 37,25% de las instancias son Malignas y 62,75% son Benignas. 

* La siguiente tabla resume el conjunto de datos:

| Propiedad | Valor |
| --- | --- |
| Clases | 2 |
| Ejemplos por clase | 212(M-0), 357(B-1) | 
| Total de instancias | 569 |
| Dimensionalidad | 30|

A continuación realizaremos las siguientes tareas:

1. Cargaremos los datos del dataset en dos variables: $x$ (features), $y$ (target).
1. Haremos un split de los datos: 500 muestras para entrenamiento (x_train, y_train) y 69 para testing (x_test, y_test)
1. Escalaremos los datos a valores entre 0 y 1.
1. Convertiremos los datos a float32 

In [ ]:
from sklearn.datasets import load_breast_cancer
import numpy as np

x, y = load_breast_cancer(return_X_y=True)

#sDivido los datos (split) en training y testing
x_train = x[:500,:]  #(500, 30)
y_train = y[:500]    #(500,) con datos 0 o 1

x_test = x[500:,:]
y_test = y[500:]

#Estandarizo los datos de entrenamiento
maxs = np.max(x_train, axis=0)  #(30,)  un valor por fila
mins = np.min(x_train, axis=0)  #(30,)  un valor por fila
M = np.max(maxs)
m = np.min(mins)
print(f'Antes --> máximo: {M}, mínimo: {m}')

x_train = (x_train - mins) / (maxs - mins)

maxs = np.max(x_train, axis=0)
mins = np.min(x_train, axis=0)
M = np.max(maxs)
m = np.min(mins)
print(f'Después --> máximo: {M}, mínimo: {m}')

#Estandarizo los datos de testeo
maxs = np.max(x_test, axis=0)
mins = np.min(x_test, axis=0)
x_test = (x_test - mins) / (maxs - mins)


#Convierto los datos a float32
print(x_train.dtype)
x_train = x_train.astype(np.float32)
print(x_train.dtype)

x_test = x_test.astype(np.float32)
y_train = y_train.astype(np.float32)
y_test = y_test.astype(np.float32)


***
* Para poder visualizar el dataset que tiene dimensión 30, usamos una herramienta conocida con el nombre de t-SNE.

* **t-SNE** (`t-Distributed Stochastic Neighbor Embedding`) es una técnica de **reducción de dimensiones** utilizada para la explotación de datos de grandes dimensiones desarrollada en 2008 por Geoffrey Hinton y Laurens Van Der Maaten. Como en el caso del Análisis de Componentes Princiaples, el objetivo es determinar un espacio de menor dimensión conservando siempre la misma distancia entre los puntos.

In [ ]:
from sklearn.manifold import TSNE
import matplotlib as mpl
import matplotlib.pyplot as plt
import random

random.seed(42)
np.random.seed(42)
mpl.rcParams['figure.figsize'] = [9.0, 6.0]

#reduzco dimensionalidad para poder visualizar el dataset 
ts_rep = TSNE().fit_transform(x_train)  

for point, label in zip(ts_rep, y_train):
    rep = 'b*' if label == 1 else 'r*'
    plt.plot([point[0]], [point[1]], rep)


Como se puede osbervar en la gráfica anterior, las dos clases de nuestro conjunto de datos pueden separarse bastante bien con una línea.

***
Ahora nos toca definir nuestro modelo (regresor logístico) y la función de pérdida (cross-entropy):

In [ ]:
import tensorflow as tf

def logisticRegresion(x, w, b):
    return 1/(1+tf.exp(-(tf.matmul(x, w) + b)))[:,0]   #matmul es equivalente a --> x @ w

def binary_crossentropy(yt, yp):
    return tf.math.reduce_mean(-yt*tf.math.log(tf.clip_by_value(yp, 1e-6, 1)) - (1-yt)*tf.math.log(tf.clip_by_value(1-yp, 1e-6, 1)))  
    #valor clipeado (recortado) para evitar log (0)

* En el cálculo de la función de pérdida, por el hecho de usar logaritmo, tenemos que prestar atención a los valores que tomará nuestra salida predicha (yp).

* Para evitar lo que se conoce como **vanishing gradient** y **exploding gradient** (que veremos más adelante cuando presentemos las redes neuronales artificiales, RNA), hay que clipear (recortar) los valores de salida del regresor logístico.

In [ ]:
w = tf.random.uniform(shape=[30, 1], minval=-1, maxval=1)  #30 features
b = tf.random.uniform(shape=[], minval=-1, maxval=1)

epochs = 100
lr = 0.1 

losses = []

for i in range(epochs):
    with tf.GradientTape() as g:
        g.watch([w, b])  
        y_hat = logisticRegresion(x_train, w, b)
        loss = binary_crossentropy(y_train, y_hat)  #calculamos la loss BCE
        losses.append(loss.numpy())

    gw, gb = g.gradient(loss, [w, b])  #calculamos los gradientes
    
    #Actualizamos ambos parámetros
    w = w - lr * gw
    b = b - lr * gb

plt.plot(losses)
plt.xlabel('epochs')
plt.ylabel('loss')
plt.title('Pérdida a lo largo del entrenamiento de un regresor logístico')

print(f'Los valores del vector w  son {w}')
print(f'El b final es {b}')

**Homework**: modificar los valores de los hiperparámetros y ver cómo cambia la función de pérdida.

***
A continuación, escribiremos una función que permita visualizar los resultados del entrenamiento mediante una **matriz de confusión**

In [ ]:
def show_confusion_matrix(cm, labels):
    '''
    Esta función grafica la matríz de confunsión (cm).
    '''
    
    fig = plt.figure()
    ax = fig.add_subplot(111)
    cax = ax.matshow(cm)

    fig.colorbar(cax)
    ax.set_xticklabels([''] + labels)
    ax.set_yticklabels([''] + labels)
    
    plt.title('Matriz de confusión')
    plt.xlabel('Predicho')
    plt.ylabel('Verdadero')
    
    for i, row in zip(range(len(cm)), cm):
        for j, val in zip(range(len(row)), row):
            ax.text(i, j, str(val), va='center', ha='center').set_backgroundcolor('white')    

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

y_pred = logisticRegresion(x_test, w, b).numpy()
# print(y_pred)

show_confusion_matrix(confusion_matrix(y_test, y_pred > 0.5), ['Maligno', 'Benigno'])  #Maligno = 0, Beningo = 1
print(classification_report(y_test, y_pred > 0.5))

`NOTAS`:

**classification_report(y_test, y_pred > 0.5))**: se utiliza la función classification_report de sklearn.metrics para generar un informe de clasificación. Esta función calcula varias métricas de evaluación, como precisión, recall, F1-score y soporte.

**confusion_matrix(y_test, y_pred > 0.5)**: la función confusion_matrix de sklearn.metrics se emplea para calcular la matriz de confusión. Se compara el vector de etiquetas verdaderas y_test con las predicciones y_pred (considerando un umbral de 0.5 para la clasificación binaria). 

En el contexto de un problema de clasificación binaria, donde se tiene un modelo de regresión logística y se busca predecir una clase binaria (por ejemplo, Maligno o Benigno), se utiliza un umbral (threshold) para tomar una decisión de clasificación. En este caso, se utiliza 0.5 como umbral.

Cuando se realiza la comparación y_pred > 0.5, se obtiene un vector de booleanos donde cada elemento indica si la predicción correspondiente es mayor que 0.5 (True) o no (False). Esto se utiliza para convertir las predicciones continuas en etiquetas binarias. Por ejemplo, si y_pred > 0.5 devuelve [True, False, True, True], significa que las primeras y últimas muestras se clasificaron como positivas (Benigno) y las demás como negativas (Maligno).

***
### Stocastic Gradient Descent

* Hasta el momento calculamos el gradiente sobre todo el conjunto de datos de entrenamiento. Sin embargo, esto puede no ser posible o eficiente cuando tenemos un conjunto de datos grande. 

* Por este motivo se suele dividir el conjunto de datos en **mini-batch** (mini-lotes) y entrenar sobre estos subconjuntos. 

* La idea es que en promedio, la agregación de los efectos de la actualización en cada mini-batch nos acerque al mínimo. Obviamente, esto no significa que cada actualización nos acerque al mínimo global.


In [ ]:
from sklearn.utils import shuffle

w = tf.random.uniform(shape=[30, 1], minval=-1, maxval=1)
b = tf.random.uniform(shape=[], minval=-1, maxval=1)

epochs = 100
lr = 0.1 
losses = []

for i in range(epochs):
    x_s, y_s = shuffle(x_train, y_train)  #mezclamos lod datos de entrenamiento    
    for mb in range(0, 500, 50):  #generamos los mini-batches (mb)
        with tf.GradientTape() as g:
            g.watch([w, b])
            y_hat = logisticRegresion(x_s[mb:mb+50], w, b)
            loss = binary_crossentropy(y_s[mb:mb+50], y_hat )
            losses.append(loss.numpy())
            
        gw, gb = g.gradient(loss, [w, b])
        w = w - lr * gw
        b = b - lr * gb
    
plt.title('Pérdida a medida que se actualiza el valor de w')
plt.xlabel('Iteraciones o actualizaciones de los parámetros')
plt.ylabel('loss')
plt.plot(losses)

* Como se puede observar, la curva de la función de pérdida no es tan suave como las anteriores, pero con los mismos hiperparámetros se logra un menor valor de pérdida.

* Miremos entonces nuestros resultados nuevamente en una matriz de confusión.

In [ ]:
y_pred = logisticRegresion(x_test, w, b).numpy()  #tensor -> ndarray
show_confusion_matrix(confusion_matrix(y_test, y_pred > 0.5), ['Maligno', 'Benigno'])
print(classification_report(y_test, y_pred > 0.5))

**HomeWork**: modifique los hiperparámetros y analice cómo varian los resultados

***
### Momentum

* Para mejorar los resultados podríamos agregar el término **momentum**. 

* La idea del *momentum* es que el algoritmo de optimización vaya llevando una historia del movimiento entre los mini-batchs. 

* A medida que baja por la función de pérdida, acumula impulso (momento), esto permite llegar más rápido cuesta abajo.

* El término momento se incluye en la actualización de los parámetros, y permite alcanzar una menor pérdida, incrementando la *accuracy* en menos epochs.

* La forma de utilizar el momentum es:

$$vel_n=momentum * vel_{n-1} – lr * grad_n$$

$$w_{n+1} = w_{n} + vel_n$$ 

In [ ]:
w = tf.random.uniform(shape=[30, 1], minval=-1, maxval=1)
b = tf.random.uniform(shape=[], minval=-1, maxval=1)

epochs = 100
lr = 0.1 
momentum = 0.9
losses = []

vw = tf.zeros(shape=[30, 1])
vb = tf.zeros(shape=[])

for i in range(epochs):
    x_s, y_s = shuffle(x_train, y_train)
    for mb in range(0, 500, 50):
        with tf.GradientTape() as g:
            g.watch([w, b])
            y_hat = logisticRegresion(x_s[mb:mb+50], w, b)
            loss = binary_crossentropy(y_s[mb:mb+50], y_hat)
        
        losses.append(loss.numpy())
        gw, gb = g.gradient(loss, [w, b])
        vw = momentum * vw - lr * gw
        vb = momentum * vb - lr * gb
        w = w + vw
        b = b + vb

plt.title('Pérdida a medida que se actualiza el valor de w')
plt.xlabel('Iteraciones o actualizaciones de los parámetros')
plt.ylabel('loss')
plt.plot(losses)

In [ ]:
y_pred = logisticRegresion(x_test, w, b).numpy()
show_confusion_matrix(confusion_matrix(y_test, y_pred > 0.5), ['Maligno', 'Benigno'])
print(classification_report(y_test, y_pred > 0.5))

`NOTAS`:

El término de **momentum** en el algoritmo SGD se utiliza para acelerar la convergencia y mejorar la estabilidad durante el proceso de optimización.

En SGD, en cada iteración se actualizan los pesos del modelo en función del gradiente de la función de costo. El término de momentum introduce una "*inercia*" en estas actualizaciones, permitiendo que se tenga en cuenta la historia de las actualizaciones anteriores.

En lugar de simplemente actualizar los pesos en función del gradiente instantáneo, el momentum agrega un componente que *tiene en cuenta la dirección y magnitud de las actualizaciones anteriores*. Esto se logra manteniendo un *promedio ponderado de las actualizaciones anteriores* y utilizando ese promedio para ajustar los pesos en la dirección del gradiente.

Un valor de momentum mayor significa que se dará más importancia a las actualizaciones anteriores, mientras que un valor menor significa que se dará más importancia al gradiente instantáneo.

Al utilizar el término de momentum, las actualizaciones de los pesos tienen una "inercia" que les permite seguir moviéndose en la misma dirección incluso si el gradiente instantáneo cambia de dirección. Esto puede ayudar a superar áreas planas o valles poco profundos en la función de costo y acelerar el proceso de convergencia.

***

### Clasificación multiclase

Para la clasificación multiclase una función muy utilizada es **softmax**. Esta función toma por entrada un vector y retorna un vector donde la suma de todos sus elementos es $1$ y todos los valores están entre **$0$** y **$1$**. 

Si nuestro problema de clasificación tiene **n clases**, podemos usar la función softmax y un vector n-dimensiones. 

Interpretaremos la salida de esta función como la distribución de probabilidades de las clases.

$$softmax(x)_{i} = \frac{e^{x_{i}}}{\sum^{n}_{j=1} e^{x_{j}}} $$



### Categorical Cross-Entropy

Esta función de pérdida considera la pérdida de información sobre la categoría real, normalizando el valor de la predicción. 

Si $\hat{y}=(\hat{y}_1, \hat{y}_2, ..., \hat{y}_C)$ es el vector de valores asociados a las clases, tenemos que:

$$P_\hat{y}=\frac{\hat{y}}{\sum\hat{y}_i}$$

$$CCE(y,P_\hat{y})=-\frac{\sum y * log(P_\hat{y})}{n} $$

NOTA: el valor de pérdida se considera **solo** sobre las **clases verdaderas**, las otras son afectadas a través de la normalización de la salida $\hat{y}$.

#### **Ejercicio B-2**: 

Para este ejercicio utilizaremos el conjunto de datos conocido como [MNIST](http://yann.lecun.com/exdb/mnist/). Dicho dataset se encuentra disponible en TensorFlow y viene dividido en entrenamiento y testing.  La tarea consiste en clasificar imagenes de dígitos escritos a mano.

| Propiedad | Valor |
| --- | --- |
| Clases | 10 |
| Tamaño de las imagenes | 28 X 28 |
| Instancias de entrenemiento | 60.000 |
| Instancias de entrenemiento | 10.000 |
| Valor mínimo de cada pixel | 0 |
| Valor máximo de cada pixel | 255 |

A continuación se carga el dataset y se dibujan los primeros 100 ejemplos del conjunto de entrenamiento.

In [ ]:
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical

(x_train, y_train), (x_test, y_test) = mnist.load_data()

print('100 primeros elementos del conjunto de entrenamiento')
f = plt.figure(111)

for i in range(10):
    for j in range(10):
        ax = f.add_subplot(10, 10, i + j*10 + 1)
        ax.set_xticklabels('')
        ax.set_yticklabels('')
        ax.imshow(x_train[i + j*10, :, :], cmap='gray')

print(y_train[:100])

Realizamos un preprocesado a los datos (reshape, normalización, one-hot encoding, cambio de tipo)

In [ ]:
print(f'x_train shape: {x_train.shape}')  #60000 x 28 x 28

size = x_train.shape[1]*x_train.shape[2] #tamaño de mi vector de entrada (hacemos un flattering a las imágenes) => 28x28 => 784
print(f'size: {size}') 
print(f'cantidad de muestras: {x_train.shape[0]}') #60000

#Reshapeamos y normalizamos ([0 , 255] --> [0, 1])
x_train = x_train.reshape((x_train.shape[0], size)) / 255  
x_test = x_test.reshape((x_test.shape[0], size)) / 255
print(f'x_train shape: {x_train.shape}')  #60000 x 784

print(y_train[0]) #digitos (target)
yc_train, yc_test = to_categorical(y_train), to_categorical(y_test)
print(yc_train[0]) #target codificado (one-hot-encode) para poder procesarlo

#Convertimos float64 a float32
x_train = x_train.astype(np.float32)
x_test = x_test.astype(np.float32)
yc_train = yc_train.astype(np.float32)
yc_test = yc_test.astype(np.float32)

Definimos la función softmax, el modelo y la función de pérdida (categorical cross-entropy):

In [ ]:
def softmax(z):
    exp = tf.exp(z)
    return exp / tf.expand_dims(tf.math.reduce_sum(exp, axis=1), axis=-1)

def predict(x, w, b):
    return softmax(tf.matmul(x, w) + b)

def categorical_crossentropy(y_true, y_pred):
    return tf.math.reduce_mean(-tf.math.reduce_sum(y_true * tf.math.log(tf.clip_by_value(y_pred, 1e-6, 1)), axis=1))

`NOTAS`:

La función **softmax(z)** implementa la función de activación softmax en TensorFlow. El softmax es comúnmente utilizado en problemas de clasificación multiclase, ya que transforma las salidas lineales (*logits*) de un modelo en *probabilidades* que representan la confianza estimada de pertenencia a cada una de las clases posibles.

**exp = tf.exp(z)**: calcula la exponencial de cada elemento de la matriz z. Esto es necesario para obtener valores no negativos y asegurar que la función softmax produzca una distribución de probabilidades válida.

**tf.math.reduce_sum(exp, axis=1)**: realiza una suma a lo largo del eje axis=1, que corresponde a las columnas de la matriz exp. Esto genera un vector que contiene la suma de los exponentes de cada fila.

**tf.expand_dims(tf.math.reduce_sum(exp, axis=1), axis=-1)**: se expande las dimensiones del vector suma para que tenga una dimensión adicional al final. Esto se hace utilizando tf.expand_dims con axis=-1, lo que agrega una dimensión de tamaño 1 al final del vector.

**exp / tf.expand_dims(tf.math.reduce_sum(exp, axis=1), axis=-1)**: se realiza la división elemento a elemento entre la matriz exp y el vector suma expandido. Cada elemento de la matriz exp se divide por el correspondiente elemento del vector suma expandido, lo que normaliza los valores y los convierte en probabilidades.

La función softmax(z) devuelve una matriz donde cada fila contiene las probabilidades de pertenencia a cada clase para el correspondiente ejemplo en z. Cada valor de la matriz está en el rango de 0 a 1, y la suma de las probabilidades de cada fila es igual a 1, lo que refleja una distribución de probabilidad válida.

La función **categorical_crossentropy(y_true, y_pred)** implementa la función de pérdida entropía cruzada categórica (categorical crossentropy) en TensorFlow, la cual se utiliza comúnmente en problemas de clasificación multiclase, donde las etiquetas de clase se representan como variables categóricas codificadas en formato *one-hot*.

**y_true**: matriz de etiquetas verdaderas codificadas en formato one-hot. Cada fila representa una muestra y cada columna representa la presencia o ausencia de la clase correspondiente. Por ejemplo, si hay 3 clases posibles y una muestra pertenece a la segunda clase, la fila correspondiente en y_true sería [0, 1, 0].

**y_pred**: matriz de predicciones del modelo, donde cada fila representa las probabilidades de pertenencia a cada clase para la correspondiente muestra. Las probabilidades deben sumar 1 en cada fila.

**tf.math.log(tf.clip_by_value(y_pred, 1e-6, 1))**: se aplica el logaritmo natural a las predicciones y_pred después de haber sido acotadas por valores mínimos y máximos. Esto se hace para evitar problemas de valores infinitos o indefinidos cuando las predicciones son muy cercanas a 0 o 1.

**y_true * tf.math.log(tf.clip_by_value(y_pred, 1e-6, 1))**: se realiza una multiplicación elemento a elemento entre y_true y el logaritmo de y_pred. Esto asegura que solo se tienen en cuenta las entradas correspondientes a las clases verdaderas en el cálculo de la pérdida.

**-tf.math.reduce_sum(...)**: efectúa la suma a lo largo del eje axis=1, que corresponde a las columnas. Esto calcula la suma de las contribuciones de pérdida para cada muestra.

**tf.math.reduce_mean(...)**: se calcula la media de las sumas de pérdida de todas las muestras en el conjunto de datos. Esto proporciona la pérdida promedio para el conjunto de datos.

El paso siguiente será entrenar nuestro modelo usando las funciones que definimos previamente.

In [ ]:
from tqdm.notebook import tqdm
from sklearn.utils import shuffle

w = tf.random.uniform(shape=[size, 10], minval=-1, maxval=1)
b = tf.random.uniform(shape=[10], minval=-1, maxval=1)

# hiperparámetros
epochs = 100
lr = 0.01 
momentum = 0.9
batch_size = 500

losses = []
vw = tf.zeros(shape=[size, 10])
vb = tf.zeros(shape=[10])

for i in tqdm(range(epochs)):  #muestra una barra de progreso a medida que van ocurriendo los epochs
    x_s, y_s = shuffle(x_train, yc_train) #target codificado
    for mb in range(0, x_s.shape[0], batch_size):
        with tf.GradientTape() as g:
            g.watch([w, b])

            y_pred = predict(x_s[mb:mb + batch_size], w, b)
            y_true = y_s[mb:mb + batch_size]

            loss = categorical_crossentropy(y_true, y_pred)
            losses.append(loss.numpy())
        
        gw, gb = g.gradient(loss, [w, b])
        vw = momentum * vw - lr * gw
        vb = momentum * vb - lr * gb
        w = w + vw
        b = b + vb
        
plt.title('Pérdida a medida que se entrena el modelo')
plt.xlabel('Iteraciones o actualizaciones de los parámetros')
plt.ylabel('loss')
plt.plot(losses)

In [ ]:
import seaborn as sn
import pandas as pd
import matplotlib.pyplot as plt

y_pred = predict(x_test, w, b).numpy()
y_pred = np.argmax(y_pred, axis=1)

array = confusion_matrix(y_test, y_pred)

df_cm = pd.DataFrame(array, index = [str(i) for i in range(10)],columns = [str(i) for i in range(10)])

sn.heatmap(df_cm, annot=True, fmt="d")

print(classification_report(y_test, y_pred))

`NOTAS`:

**y_pred = predict(x_test, w, b).numpy()**: llama a la función predict para obtener las predicciones del modelo utilizando el conjunto de prueba x_test y los parámetros w y b. Luego, se utiliza el método .numpy() para obtener un arreglo NumPy con los valores de las predicciones. Esto permite trabajar con los resultados como un arreglo de NumPy en lugar de un tensor de TensorFlow.

**y_pred = np.argmax(y_pred, axis=1)**: utiliza la función np.argmax de NumPy para obtener los índices de los valores máximos en cada fila del arreglo y_pred. El parámetro axis=1 indica que se debe buscar el máximo en cada fila (clase), en lugar de en cada columna (muestra). Esto devuelve un arreglo de NumPy con los índices de las clases predichas para cada muestra.

En resumen, el código toma las predicciones continuas y_pred generadas por el modelo y las transforma en etiquetas de clase discreta utilizando np.argmax. Cada valor en y_pred representa las probabilidades de pertenencia a cada clase, y np.argmax selecciona la clase con la probabilidad más alta como la predicción final. El resultado se almacena en y_pred para su posterior uso o evaluación.


En el contexto de la evaluación de clasificadores, las métricas:  precisión (precision), exactitud (accuracy), recuperación (recall) y puntuación F1 (F1-score), son utilizadas para analizar los resultados y evaluar el rendimiento de un clasificador en un problema de clasificación multiclase, como el conjunto de datos MNIST.

**Precisión (precision)**: es la proporción de instancias clasificadas correctamente como positivas (verdaderos positivos) en relación con todas las instancias clasificadas como positivas (verdaderos positivos más falsos positivos). La precisión mide la capacidad del clasificador para evitar clasificar incorrectamente una instancia negativa como positiva. Una alta precisión indica que el clasificador tiene menos falsos positivos.

**Exactitud (accuracy)**: es la proporción de instancias clasificadas correctamente en relación con el total de instancias. Es una medida general del rendimiento del clasificador y representa la proporción de predicciones correctas en todas las clases. La exactitud es una métrica útil cuando las clases están equilibradas en el conjunto de datos.

**Recuperación (recall)**: también conocida como *sensibilidad o tasa de verdaderos positivos (TPR)*, es la proporción de instancias positivas correctamente clasificadas (verdaderos positivos) en relación con todas las instancias positivas (verdaderos positivos más falsos negativos). La recuperación mide la capacidad del clasificador para encontrar todas las instancias positivas y es `útil cuando el énfasis está en minimizar los falsos negativos`.

**Puntuación F1 (F1-score)**: es una medida que combina la precisión y el recall en un solo valor. Es la media armónica entre ambas métricas y proporciona una medida equilibrada del rendimiento del clasificador. Es útil cuando se desea encontrar un equilibrio entre la precisión y la recuperación.

**Soporte (support)**: es el número de instancias en cada clase. Representa la cantidad de muestras en el conjunto de datos que pertenecen a cada clase y se utiliza para calcular las métricas anteriores.

El soporte te dará información sobre la distribución de las clases en el conjunto de datos. Esto te permitirá evaluar el rendimiento del clasificador en la tarea de clasificación de dígitos escritos a mano y comprender en qué aspectos se destaca o puede haber áreas de mejora.